# Subsample datasets to 100 per year

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import random
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [15]:
# Set up directories

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/complete_human/"

references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

# os.chdir(references)
# states_ref = pd.read_csv("states_ref.csv")

## Upload FASTAs

In [16]:
# Organize fastas

fastas = {}
for dirpath, dirs, files in os.walk(home + "to_subsample/"):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = df_from_fasta(file_name)
            fastas[file_name.split("/")[-1]] = fasta
    break # Do not go into subfolders

In [17]:
print(fastas)

{'europe_human+swine_pdmOnly_2009-2026_aln_trim_downsampled.fasta':                                             full_header  \
0     >MZ945846|A/swine/Spain/45690-9/2018|H1N2|Spai...   
1     >MW848689|A/swine/Spain/31001-2/2019|H1N1|Spai...   
2     >PP331791|A/swine/Spain/05165-1/2020|H1N1|Spai...   
3     >PP338521|A/swine/Spain/44593-1/2021|H1N1|Spai...   
4     >PQ107580|A/swine/Spain/44394-1/2021|H1N1|Spai...   
...                                                 ...   
8671  >EPI_ISL_20347483|A/England/01898391/2026|H1N1...   
8672  >EPI_ISL_20314783|A/Netherlands/10010/2026|H1N...   
8673  >EPI_ISL_20314781|A/Netherlands/10009/2026|H1N...   
8674  >EPI_ISL_20314766|A/Netherlands/10001/2026|H1N...   
8675  >EPI_ISL_20314765|A/Netherlands/10000/2026|H1N...   

                                               sequence  
0     atgaaggcaatactagtagttctgctatatacatttgcgaccacaa...  
1     atgaaggcaataatggtagttctgctatatacatttgcaaccgcag...  
2     atgaaggcaatactagtagttctgctgtatacatttacaaccg

## Subsample

In [20]:
# Create a dictionary of dictionaries of dataframes grouped by year

fastas_grouped = {} # Overall dictionary -- length is number of datasets needed

for key in fastas:
    years = {} # For each dataset, there are a number of years
    fasta = fastas[key]
    fasta["Year"] = fasta["full_header"].apply(lambda x: dateutil.parser.parse(x.split("|")[4]).year) # Find year
    fasta["Isolate"] = fasta["full_header"].apply(lambda x: x.split("|")[1].lower()) # if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
    fasta = fasta.drop_duplicates(subset="Isolate", keep="first")
    for year, rows in fasta.groupby("Year"): # Separate dataframe into multiple dataframes by year
        years[year] = rows # For each year, there are a number of entries that have that year
    fastas_grouped[key] = years

print(fastas_grouped)

{'europe_human+swine_pdmOnly_2009-2026_aln_trim_downsampled.fasta': {2009:                                             full_header  \
12    >EPI_ISL_73603|A/swine/Italy/85437/2009|H1N1|I...   
105   >EPI_ISL_195175|A/swine/England/1353/2009|H1N1...   
119   >EPI_ISL_129507|A/swine/England/MD0040352R/200...   
292   >EPI_ISL_103101|A/swine/Norway/02_11342/2009|H...   
339   >EPI_ISL_66077|A/swine/Italy/290271/2009|H1N1|...   
...                                                 ...   
4873  >EPI_ISL_60232|A/Estonia/2/2009_BT|H1N1|Estoni...   
4880  >EPI_ISL_93449|A/Paris/7973/2010|H1N1|Paris|20...   
4881  >EPI_ISL_93451|A/Paris/7975/2010|H1N1|Paris|20...   
4882  >EPI_ISL_60228|A/Latvia/6-1221/2009v|H1N1|Latv...   
4884  >EPI_ISL_93450|A/Paris/7974/2010|H1N1|Paris|20...   

                                               sequence  Year  \
12    atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...  2009   
105   atgaaggcaatactaatagttctgctatatacatttgcaaccgcaa...  2009   
119   atgaaggcaatacta

In [21]:
subsampled_dfs = {} # Overall subsampled dictionary -- length is number of datasets needed
for key in fastas_grouped:
    dataset = fastas_grouped[key] # Dataset
    df = pd.DataFrame() # Hold subsampled data
    for year_key in dataset: # Dictionary of years and their dataframes 
        year_df = dataset[year_key] # One year and its data
        # If there are more than 100 entries, subsample a random 100 
        subsampled = year_df[["full_header", "sequence"]].sample(n=100, random_state=2008) if len(year_df) > 100 else year_df[["full_header", "sequence"]]
        # print(subsampled)
        df = pd.concat([df, subsampled]) # Add subsampled data to dataframe
    subsampled_dfs[key] = df # Add dataframe to dictionary of datasets

print(subsampled_dfs)
        

{'europe_human+swine_pdmOnly_2009-2026_aln_trim_downsampled.fasta':                                             full_header  \
1786  >EPI_ISL_100747|A/Austria/07/2009|H1N1|Austria...   
907   >EPI_ISL_65989|A/ENG/483/2009|H1N1|ENG|2009-06...   
1917  >EPI_ISL_67944|A/ENG/676/2009|H1N1|ENG|2009-07...   
876   >EPI_ISL_66472|A/Moldova/G-104/2009|H1N1|Moldo...   
1686  >EPI_ISL_101433|A/England/395/2009|H1N1|Englan...   
...                                                 ...   
8388  >EPI_ISL_20326904|A/Netherlands/10015/2026|H1N...   
8470  >EPI_ISL_20339304|A/Netherlands/10030/2026|H1N...   
8644  >EPI_ISL_20347745|A/England/01898454/2026|H1N1...   
8509  >EPI_ISL_20342425|A/Belgium/S0977/2026|H1N1|Be...   
8406  >EPI_ISL_20331773|A/England/01894400/2026|H1N1...   

                                               sequence  
1786  atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...  
907   atggaggcaatactagtagttctgctatatacatttgcaaccgcaa...  
1917  atgaaggcaatactagtagttctgctatatacatttgcaaccg

## Assemble FASTAs

In [22]:
# for key in subsampled_dfs:
#     fasta_df = subsampled_dfs[key]
#     fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
#     fasta_df["Host"] = fasta_df["full_header"].apply(lambda x: "human" if "human" in x else x.split("/")[1] if "/" in x else "unknown") #  if len(x.split("/")) > 1 else "unknown")
#     fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2] if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
#     fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
#     fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
#     fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
#     fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
    
#     print(fasta_df)

## Download FASTAs

In [23]:
# Prepare for download
for key in subsampled_dfs:
    file_name = "subsampled_" + key # Create file name
    fasta = subsampled_dfs[key]
    df_to_fasta(fasta, file_name, home)
